In [1]:
import sys, os
import pandas as pd
sys.path.append(os.path.abspath('..'))
%load_ext autoreload
%autoreload 2

In [2]:
from src.data_cleaning import load_raw_data, clean_data, save_processed_splits
from src.feature_engineering import engineer_features, target_encode_column
from sklearn.model_selection import train_test_split

df_raw = load_raw_data()
df = clean_data(df_raw)
df = engineer_features(df)
df.dtypes

الحي                    object
الشارع                  object
الطابق                   int64
عدد الغرف                int64
عدد الحمامات             int64
المساحة (م2)             int64
الوجهة                   int64
الإكساء                  int64
عدد الشقق في كل طابق     int64
موقف سيارة               int64
وجود مصعد                int64
السعر                    int64
اتجاه_شرقي                bool
اتجاه_شمالي               bool
اتجاه_غربي                bool
dtype: object

## Train/test split


In [3]:
feature_cols = [c for c in df.columns if c not in ['السعر', 'الحي', 'الشارع']]
X = df[feature_cols + ['الشارع', 'الحي']]
y = df['السعر']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

Train: 113 | Test: 20


## Correlation & mutual information (training data only)

In [4]:
train_with_price = X_train.copy()
train_with_price['السعر'] = y_train
train_with_price.corr(numeric_only=True)['السعر'].sort_values(ascending=False)

السعر                   1.000000
المساحة (م2)            0.852382
عدد الحمامات            0.850510
عدد الغرف               0.839575
الوجهة                  0.659135
موقف سيارة              0.494724
اتجاه_غربي              0.333721
الإكساء                 0.286521
اتجاه_شمالي             0.115161
وجود مصعد               0.060185
الطابق                  0.049288
اتجاه_شرقي             -0.144977
عدد الشقق في كل طابق   -0.447719
Name: السعر, dtype: float64

In [5]:
train_with_price.groupby('الطابق')['السعر'].mean().sort_index()

الطابق
-1     72333.333333
 0    101571.428571
 1     98272.727273
 2    106923.076923
 3    109800.000000
 4     82615.384615
Name: السعر, dtype: float64

In [6]:
from sklearn.feature_selection import mutual_info_regression

numeric_features = [c for c in X_train.columns if c not in ['الشارع', 'الحي']]
discrete_cols = ['الطابق', 'الإكساء', 'الوجهة', 'موقف سيارة', 'وجود مصعد',
                  'اتجاه_شمالي', 'اتجاه_شرقي', 'اتجاه_جنوبي', 'اتجاه_غربي']
discrete_mask = [c in discrete_cols for c in numeric_features]

mi_scores = mutual_info_regression(X_train[numeric_features], y_train,
                                    discrete_features=discrete_mask, random_state=42)
pd.Series(mi_scores, index=numeric_features).sort_values(ascending=False)

المساحة (م2)            0.669475
عدد الغرف               0.593070
عدد الحمامات            0.436098
الوجهة                  0.298215
موقف سيارة              0.156807
عدد الشقق في كل طابق    0.137076
وجود مصعد               0.077230
اتجاه_شمالي             0.074438
الإكساء                 0.053085
اتجاه_غربي              0.049016
اتجاه_شرقي              0.031507
الطابق                  0.030339
dtype: float64

## Street target encoding + final cleanup

In [7]:
X_train = X_train.drop(columns=['الحي'])
X_test = X_test.drop(columns=['الحي'])

X_train, X_test, street_map = target_encode_column(X_train, X_test, y_train, column='الشارع', smoothing=10)

bool_cols = X_train.select_dtypes(include='bool').columns
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

street_map

الشارع
الأنوار     94772.056852
الدرويش    103689.422672
الفندق     106203.913748
النورس     102498.298162
وردشان      86499.115044
dtype: float64

In [8]:
import joblib
save_processed_splits(X_train, X_test, y_train, y_test)
joblib.dump(street_map, '../models/street_encoding_map.pkl')
joblib.dump(y_train.mean(), '../models/global_mean_price.pkl')
print("Saved processed splits and encoding artifacts.")

Saved processed splits and encoding artifacts.
